# Download the dataset


# Confident Legal Research Application

- Main stakeholder/consumer: Junior Legal Assistant
- Distributed: No


**Goal:Improve quality & speed of case research for junior assistants**

## Baseline System

To create evaluation process we would need to have some baseline system in place. That is why we are creating a **Naive implementation** with the following initial setup:

- One sparse vector embedding (simplest 384 encoder)
- Search function with no reranker
- Naive RAG pipeline

In [2]:
%load_ext autoreload
%autoreload 2

### Initialize dataset

We will use famouse CLERC dataset for this tutorial. Two reasons:
1. There exist queries in the dataset that we can use as a baseline for complex law queries 
2. The combination of queries/positive/negative responses provides a good baseline for the *golden set* that could be used for testing

In [12]:
from src.dataset import load_clerc


clerc_data = load_clerc("train", limit=2000)

clerc_data

Loading dataset shards:   0%|          | 0/31 [00:00<?, ?it/s]

Dataset to pydantic transformation:   0%|          | 0/2000 [00:00<?, ?it/s]

ClercSplit(corpus=41167 docs, queries=2000, qrels=2000)

In [13]:
sample_tuple = clerc_data.sample()

sample_tuple

(Query(query_id='121522', query='determining whether to reconsider claims: § 502(j) is supplemented by Federal Rule of Bankruptcy Procedure 3008, which states: “A party in interest may move for reconsideration of an order allowing or disallowing a claim against the estate. The court after a hearing on notice shall enter an appropriate order.” FED. R. BANKR. P. 3008. The phrase “for cause,” as it is used in § 502(j), is not specifically defined in either the Bankruptcy Code or the Rules, “but is an adaptable standard reflecting bankruptcy laws’ roots in equity jurisprudence.” See Advisory Committee’s Notes to FED. R.BANKR. P. 3008. Thus, the bankruptcy court possesses broad discretion in determining whether adequate cause exists for the reconsider ation of claims. See  REDACTED In re Lomas Financial Corp., 212 B.R. 46, 52 (Bankr.D.Del.1997). In re Bernardes, 267 B.R. 690, 693 (Bankr.D.N.J.2001). Moreover, although “cause” is not expressly defined in- the Bankruptcy Code, “courts have su

### Initialize vectors 

- use `qdrant_client` for storing vectors
- use `fastembed` for the simplest embeddings generation pipelines


We will also store the embeddings locally and try to handle them paralelly 

>NB!: This notebook is optimized to run on macbook devices. Try to adapt embedding config to your specific hardware

In [14]:
from qdrant_client import QdrantClient
import os

qdrant_url = os.getenv('QDRANT_URL')
qdrant_api_key = os.getenv('QDRANT_API_KEY')

client = QdrantClient(url=qdrant_url, api_key=qdrant_api_key)

In [15]:
# constants
COLLECTION_NAME = 'lawyer_citations'
DENSE_MODEL = "BAAI/bge-small-en-v1.5"
SIZE = 384

In [16]:
from src.indexing import EmbeddingConfig
from qdrant_client.models import Distance


config = EmbeddingConfig(
    name='dense_base',
    model_id=DENSE_MODEL,
    kind='dense',
    size=SIZE,
    distance=Distance.COSINE,
    providers=["CoreMLExecutionProvider", "CPUExecutionProvider"],
    parallel=None,
)

In [17]:
from src.indexing import EmbeddingCache
embedding_cache = EmbeddingCache()

In [18]:
from src.indexing import DocumentIndexer

document_vectors = DocumentIndexer(
    client,
    COLLECTION_NAME,
    embeddings=[config],
    cache=embedding_cache
)

In [19]:
document_vectors.ensure_collection()

In [20]:
document_vectors.upload(items=list(clerc_data.corpus.values()), batch_size=128)

embed:dense_base:   0%|          | 0/41167 [00:00<?, ?it/s]

In [23]:
client.get_collection(COLLECTION_NAME).points_count, (client.get_collection(COLLECTION_NAME).points_count or 0) > 0

(41167, True)

### Create simple semantic search and RAG pipelines

- `DenseSearcher` is the simplest implementation of the semantic search: no sparse vectors, multivectors, no reranking. Just query API
- `LegalAssistant` - uses `DenseSearcher` for the retrieval and generates the results based on that. We use `anthropic api` with `lite-llm`

In [25]:
from src.search import DenseSearcher
from src.rag import LegalAssistant

qa = clerc_data.sample()
searcher = DenseSearcher(client, COLLECTION_NAME, config)
legal_assistant = LegalAssistant(searcher)


In [27]:
result = searcher.search(qa[0].query)
result, qa[1]

([SearchHit(doc_id='11591376', text='we held that the officers lawfully arrested him because he “ ‘voluntarily exposed himself to warrantless arrest’ by freely opening the door of his motel room to the police.” Id. at 1426 (quoting United States v. Johnson, 626 F.2d 753, 757 (9th Cir.1980)). Unlike the officers in Jardines and in this case, the officers in Vaneaton were standing in the common space of a motel when they knocked, rather than in the curtilage of a home. We therefore have no need to overrule Vaneaton. See Miller v. Gammie, 335 F.3d 889, 899-900 (9th Cir.2003) (en banc) (holding that “a three- judge panel is free to reexamine the holding of a prior panel” when the Supreme Court has “undercut the theory or reasoning underlying the prior circuit precedent in such a way that the cases are clearly irreconcilable”). Whether Vaneaton remains good law after Jardines is therefore a question for another case and another day. B. Protective Sweep The protective sweep doctrine authoriz

In [29]:
answer = legal_assistant.ask("My clients landlord wants him to pay for the wall that was broken. No hand-in protocol was signed. Give me cases with similar problems")

print(answer)

Based on the case excerpts provided, none of them directly address the specific legal issue of a landlord seeking payment for property damage (a broken wall) in the absence of a signed hand-in (or check-in/check-out) protocol. The excerpts deal with unrelated legal matters, including bankruptcy and furniture repossession, farm rent assignments, government contract disputes, emergency rent control regulations, and bankruptcy court jurisdiction. None of these cases involve a landlord-tenant dispute over property damage or the evidentiary significance of a missing hand-in protocol.

Therefore, the provided excerpts do not contain sufficient information to answer your question about cases involving a landlord seeking compensation for property damage where no hand-in protocol was signed. You would need to consult additional case law specifically addressing landlord-tenant property damage disputes and the legal effect of the absence of a move-in/move-out inspection report or protocol.

Citat

### Basis of the evaluation framework 

From the first test query its obvious - that our retrival layer is not doing particurarly good job - we explicitly tried to make a PoC version with simplest setup, to showcase what can happen in reality, even in more complex systems. 

The good thing here is that LLM itself is smart enough (we are using quite smart model for testing - `claude-sonnet-4.6` is strong model from antropic) to understand that the queries do not make much sense. However, in production that might be not true again, issues might be hidden, which might lead to a cituation where both unopinionated domain experts and llm models wont help. 

## Goal oriented evaluation specification

Lets try to create some evaluation that will make sense for the current system. We as it was mentioned, start **top-down** - there is no reason to use the system if its business goal is not satisfied. 

### Inputs/Assumptions. What do we know


First lets start with the inputs that we know of. These are all the preconditions that we will feed into our methodology to go down the decision tree. So far they are the following:
- **GOAL:** Confident Legal Research
- **TARGET/CONSUMER:** Junior Assisstant (not very law rigourus, needs guidance and explanations)
- **IS DISTRIBUTED:** No, system had one goal, one consumer

In [33]:
from IPython.display import HTML

HTML("""
<div style="text-align: center;">
    <img src="diagrams/eval-inputs.svg" width="300">
</div>
""")

### Goal Decomposition

A bridge between high level goal like the one specified and the actual metrics is a process called goal decomposition. We have to know what exactly the system will look like and what requirements does it have to be able to quantify it.


Thus, we have to answer the following questions: 
- Is this a *business goal*? Yes, because we are not describing the system we are describing its utility 
- Can we make *component level assumptions*?  Yes, we have two components **searcher** which is system retrieval and we know how it works, and **rag** which is a llm driven component


Next lets specify what exactly does the business want? A more refined version of that would be:

`Goal as business objective: Improve quality & speed of case research for junior assistants`

From the previous code base we can also say that the technical team made certain architectural decisions. Lets write them down as well in a general sense:

`Tech goal: Confident & Fast Legal Agentic Search (too vague to test)`


Now we can do goal decomposition. This technique is one of the oldest design principles in SE, and its something we can use to specify our context for the evaluations:

![Goal Decomposition](diagrams/goal-decomposition.svg)